### Vectorless RAG (PageIndex)

Traditional vector-based RAG splits a document into chunks, embeds them, stores them in a vector database, and retrieves similar chunks using semantic similarity search. This approach can struggle with chunk boundaries and requires maintaining embeddings.

**Vectorless RAG** takes a different approach: instead of embeddings and similarity search, it builds a hierarchical tree index that mirrors the document's natural structure (like a table of contents), and uses an LLM to *reason* over the tree to find relevant sections—much like a human flipping through a book's index. This eliminates the need for embedding models and vector databases, but requires documents with clear structure and incurs extra LLM calls at index-build time.

### Setup

We'll use the **PageIndex** OSS package to build and query the tree, and reuse the `OPENAI_API_KEY` already in your `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Sample document

We'll use the classic "Attention Is All You Need" paper (arXiv:1706.03762) as our sample document. It's a short, freely available paper with a clear hierarchical structure (Abstract → Introduction → Background → Model Architecture with subsections 3.1–3.5 → Training → Results → Conclusion), making it ideal for demonstrating tree-based section retrieval.

In [ ]:
import requests

os.makedirs("data", exist_ok=True)
PDF_PATH = "data/attention_is_all_you_need.pdf"

if not os.path.exists(PDF_PATH):
    print("Downloading 'Attention Is All You Need' paper...")
    resp = requests.get("https://arxiv.org/pdf/1706.03762")
    with open(PDF_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded to {PDF_PATH}")
else:
    print(f"PDF already exists at {PDF_PATH}")

PDF_PATH

### Building the PageIndex tree

PageIndex reads the PDF and uses an LLM to detect section boundaries, headings, and content structure. It builds a JSON tree where each node has a title, page range, and (optionally) a generated summary of what that section covers. This tree replaces the embedding step in traditional vector RAG—it's the document's structural index.

In [ ]:
from pageindex import PageIndex

# Initialize PageIndex and build the tree from the PDF
page_index = PageIndex(PDF_PATH)
tree = page_index.build_tree()

print(f"Tree built with {len(tree.get_all_nodes())} nodes")
tree

### Inspecting the tree structure

The tree mirrors the document's table of contents. Each node knows its section title, which pages it spans, and what content it covers—all without any vector embeddings.

In [ ]:
def print_tree(node, indent=0):
    """Recursively print the tree structure."""
    prefix = "  " * indent + "├─ "
    title = node.get("title", "(untitled)")
    page_range = node.get("page_range", "?")
    node_id = node.get("id", "?")
    print(f"{prefix}{title} [id: {node_id}, pages: {page_range}]")
    
    for child in node.get("children", []):
        print_tree(child, indent + 1)

print("Document Structure:")
print_tree(tree.root)

### Querying: LLM reasoning over the tree

Instead of embedding the user's question and doing similarity search, PageIndex passes the tree structure (node titles/summaries) plus the question to an LLM. The LLM reasons about which section(s) are relevant—like a human scanning a book's index to find the answer.

In [ ]:
# Ask a question that we know lives in a specific section (Multi-Head Attention, section 3.2.2)
question = "How does the paper compute multi-head attention?"

# Retrieve relevant nodes using PageIndex's reasoning
retrieval_result = page_index.retrieve(question, top_k=3)

print(f"Question: {question}")
print(f"\nSelected nodes:")
for node_info in retrieval_result:
    print(f"  - {node_info.get('title')} (score: {node_info.get('score', 'N/A')})")

if retrieval_result and "reasoning" in retrieval_result[0]:
    print(f"\nLLM reasoning trace:")
    print(retrieval_result[0]["reasoning"])

### Retrieving the section text

Once PageIndex has identified the relevant node(s), we extract the full text of that section from the PDF. We get the whole, coherent section—not scattered chunks split across boundaries.

In [ ]:
from pypdf import PdfReader

# Get the top-scoring retrieved node
top_node = retrieval_result[0]
page_range = top_node.get("page_range", [0, 1])
start_page, end_page = page_range[0], page_range[1]

# Extract text from the PDF for this page range
reader = PdfReader(PDF_PATH)
retrieved_text = ""
for page_num in range(start_page, min(end_page + 1, len(reader.pages))):
    retrieved_text += reader.pages[page_num].extract_text()

print(f"Retrieved text from '{top_node['title']}' (pages {start_page}-{end_page}):")
print(f"\n{retrieved_text[:500]}...\n")  # Print first 500 chars as a preview

### Generating the final answer

Now we use LangChain's `init_chat_model()` to pass the retrieved section as context and generate an answer grounded in the actual document content.

In [ ]:
from langchain.chat_models import init_chat_model

# Initialize the answer-generation model
answer_model = init_chat_model("openai:gpt-4o-mini")

# Build the prompt with the retrieved context
context_prompt = f"""Answer the following question using ONLY the context provided below. 
Do not use any external knowledge.

Context:
{retrieved_text}

Question: {question}

Answer:
"""

# Generate the answer
response = answer_model.invoke(context_prompt)

print(f"Answer: {response.content}")

### Testing discrimination with a second question

Let's ask a different question that targets a completely different section of the paper. This demonstrates that PageIndex's tree-based reasoning actually discriminates between sections rather than always returning the same nodes.

In [ ]:
# Ask a question targeting the Training section (section 5)
question_2 = "What optimizer and learning rate schedule did they use for training?"

# Retrieve relevant nodes
retrieval_result_2 = page_index.retrieve(question_2, top_k=3)

print(f"Question: {question_2}")
print(f"\nSelected nodes:")
for node_info in retrieval_result_2:
    print(f"  - {node_info.get('title')} (score: {node_info.get('score', 'N/A')})")

# Extract and answer using this retrieval
top_node_2 = retrieval_result_2[0]
page_range_2 = top_node_2.get("page_range", [0, 1])
start_page_2, end_page_2 = page_range_2[0], page_range_2[1]

reader = PdfReader(PDF_PATH)
retrieved_text_2 = ""
for page_num in range(start_page_2, min(end_page_2 + 1, len(reader.pages))):
    retrieved_text_2 += reader.pages[page_num].extract_text()

context_prompt_2 = f"""Answer the following question using ONLY the context provided below.

Context:
{retrieved_text_2}

Question: {question_2}

Answer:"""

response_2 = answer_model.invoke(context_prompt_2)
print(f"\nAnswer: {response_2.content}")

### Comparing to vector-based RAG

**Traditional Vector RAG Pipeline:**
1. Load PDF with `PyPDFLoader`
2. Split into chunks using `RecursiveCharacterTextSplitter`
3. Embed chunks using an embeddings model (e.g., OpenAI's `text-embedding-3-small`)
4. Store embeddings in a vector store (e.g., Chroma, FAISS, Pinecone)
5. On query, embed the question and retrieve top-k similar chunks via vector similarity search
6. Stuff the top-k chunks into the LLM prompt and generate an answer

**Vectorless RAG (PageIndex) Trade-offs:**
- **Advantage:** No embeddings or vector DB needed. Retrieves whole, coherent sections, not arbitrary chunks. Provides an inspectable reasoning trace showing *why* a section was selected.
- **Disadvantage:** Requires documents with clear hierarchical structure. Extra LLM calls at tree-build time. May not work well for unstructured or poorly-formatted documents.

Vectorless RAG shines for well-structured documents (research papers, technical reports, book chapters) where the document's natural sections align with user questions. Vector RAG is better for unstructured text corpora where semantic similarity (not structural proximity) matters most.

### Aside: PageIndex Cloud API

PageIndex also offers a hosted Cloud API (https://pageindex.ai). Instead of running the OSS package locally, you can upload a PDF via REST and get back the same tree index and retrieval endpoints. This requires a separate `PAGEINDEX_API_KEY` and account. For this learning notebook we've used the fully self-contained OSS approach, but the Cloud API is useful if you prefer managed hosting or have large-scale indexing needs.